# From Cosmological Parameters to Matter Power Spectra with PyCCL

This notebook introduces [PyCCL](https://ccl.readthedocs.io/) (the Core Cosmology Library), a Python package for computing cosmological quantities with validated numerical accuracy.

## The path through the notebook

- Set up `pyccl`, `numpy`, and `matplotlib`
- Turn a set of cosmological parameters into a reusable CCL cosmology
- Translate observed redshift $z$ into the scale factor $a$ expected by CCL
- Use a distance calculation as a first physical check
- Build numerical grids with NumPy
- Compute and compare linear and non-linear matter power spectra $P(k)$

The argument moves from inputs to physical predictions. At the end, two assignments
ask you to apply the same workflow to distance measures and power spectra at several
redshifts.

**Useful documentation:** [PyCCL quickstart](https://ccl.readthedocs.io/en/latest/source/quickstart.html) · [API reference](https://ccl.readthedocs.io/en/latest/api/)

---

## Set up the computational tools

We will use three libraries throughout this notebook:

| Package | Role |
|---------|------|
| `pyccl` | Cosmology calculations (distances, power spectra, …) |
| `numpy` | Numerical arrays and grids (`linspace`, `logspace`, …) |
| `matplotlib.pyplot` | Plotting figures |

Run the cell below and check that PyCCL prints a version number (e.g. `3.2.1`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy
import pyccl
from matplotlib import pyplot

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this notebook from within the IAFlowCloud repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / "Code"
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))
data_path = PROJECT_ROOT / "Data" / "General"
figure_path = PROJECT_ROOT / "Figure" / "General" / "CCL"
figure_path.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)


---

## Turn parameters into a reusable cosmology

Almost every calculation in PyCCL starts from a `Cosmology` object. This object stores the cosmological parameters and caches derived quantities (distances, power spectra, …) as you request them.

The values read from `Planck.json` describe a Planck-like cosmology. With
$w_0=-1$, $w_a=0$, and $\Omega_k\simeq0$, the model is effectively flat
$\Lambda$CDM. The table below lists every argument used in the following
`Cosmology(...)` call.

### Cosmological parameters read from `Planck.json`

| CCL argument | JSON key | Current value | Meaning |
|---|---|---:|---|
| `h` | `H` | $0.6732$ | Reduced Hubble parameter, with $H_0=100h\,\mathrm{km\,s^{-1}\,Mpc^{-1}}$ |
| `w0` | `W0` | $-1$ | Present-day dark-energy equation-of-state parameter $w_0$ |
| `wa` | `WA` | $0$ | Time-evolution parameter in $w(a)=w_0+w_a(1-a)$ |
| `A_s` | `AS` | $2.101\times10^{-9}$ | Amplitude of the primordial scalar power spectrum |
| `n_s` | `NS` | $0.966$ | Spectral index of the primordial scalar power spectrum |
| `m_nu` | `MNU` | $0.06\,\mathrm{eV}$ | Sum of neutrino masses |
| `T_CMB` | `TCMB` | $2.7255\,\mathrm{K}$ | Present-day CMB temperature |
| `Omega_k` | `OMEGAK` | $-1.89\times10^{-7}$ | Spatial-curvature density parameter today |
| `Omega_c` | `OMEGAC` | $0.265029$ | Cold-dark-matter density parameter today |
| `Omega_b` | `OMEGAB` | $0.04939$ | Baryon density parameter today |

The JSON file contains additional derived or alternative quantities such as
`SIGMA8`, `OMEGAM`, and `NEFF`, but the cell below does not pass them to CCL.

### Fixed CCL and CAMB settings

| Argument or option | Value | Meaning |
|---|---|---|
| `mass_split` | `'normal'` | Use the normal neutrino-mass hierarchy |
| `transfer_function` | `'boltzmann_camb'` | Compute the transfer function with CAMB |
| `camb.kmax` | `100` | Maximum CAMB wavenumber used internally |
| `camb.lmax` | `5000` | Maximum CAMB multipole |
| `camb.halofit_version` | `'mead2020_feedback'` | Use the Mead 2020 feedback version of HMCode/Halofit |
| `camb.HMCode_logT_AGN` | `7.8` | Baryonic-feedback heating-temperature parameter |

We will reuse the same `cosmology` object in all later cells.

In [ ]:
with open(data_path / 'Planck.json', 'r') as file:
    parameter = json.load(file)
print(parameter)

In [ ]:
cosmology = pyccl.cosmology.Cosmology(
    h = parameter['H'],
    w0 = parameter['W0'],
    wa = parameter['WA'],
    A_s = parameter['AS'], 
    n_s = parameter['NS'], 
    m_nu = parameter['MNU'], 
    T_CMB = parameter['TCMB'],
    Omega_k = parameter['OMEGAK'], 
    Omega_c = parameter['OMEGAC'], 
    Omega_b = parameter['OMEGAB'], 
    mass_split='normal', transfer_function = 'boltzmann_camb', 
    extra_parameters = {'camb': {'kmax': 100, 'lmax': 5000, 'halofit_version': 'mead2020_feedback', 'HMCode_logT_AGN': 7.8}}
)

print(cosmology)

---

## Translate redshift into scale factor

In cosmology, the **scale factor** $a$ describes the relative size of the Universe. By convention, $a = 1$ today.

The cosmological **redshift** $z$ is related to the scale factor by

$$
a = \frac{1}{1 + z}.
$$

Examples:
- Today: $z = 0$ $\Rightarrow$ $a = 1$
- At $z = 1$: $a = 0.5$
- At $z = 2$: $a = 1/3$

**Important for PyCCL:** many functions take the scale factor `a` as an argument, not redshift. Whenever you are given a redshift, convert it with `a = 1.0 / (1.0 + z)` before calling CCL.

### Check the distance convention at the present day

As a sanity check, we compute the **comoving radial distance** to $z = 0$ (i.e. to today). The result must be zero, because we are "looking" at our own location.

The function we use is `pyccl.comoving_radial_distance(cosmo, a)`. Later, in the **distance assignment**, you will compute several related distance measures over a range of redshifts and plot them.

In [ ]:
# Quick sanity check: comoving distance at z = 0

z0 = 0.0
a0 = 1.0 / (1.0 + z0)
chi0 = pyccl.comoving_radial_distance(cosmology, a0)
print(f"Comoving distance to z={z0}: {chi0:.2f}Mpc")

---

## Build the numerical grids

Before computing power spectra, we briefly recap a few NumPy tools. In scientific Python we almost always work with **arrays**, not plain Python lists, because:

- arrays support element-wise arithmetic
- they are faster for large grids
- libraries like PyCCL and Matplotlib expect array-like inputs

The next cells show:
1. converting a list to an array
2. basic statistics (`.mean()`, `.std()`, `.min()`, `.max()`)
3. building evenly spaced grids with `numpy.arange` and `numpy.linspace`
4. building a **logarithmic** $k$-grid with `numpy.logspace` (standard for power spectra)

In [ ]:
value_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
value_array = numpy.array(value_list)
print(value_list, value_array)

In [ ]:
value_array.mean(), value_array.std(), value_array.min(), value_array.max()

### Build regular grids

- `numpy.arange(start, stop, step)` — values from `start` up to (but not including) `stop`, with a fixed step
- `numpy.linspace(start, stop, num)` — `num` points equally spaced between `start` and `stop` (inclusive if `endpoint=True`)

You will use something like `linspace` in the distance assignment to build a redshift array.

In [ ]:
array = numpy.arange(start=0, stop=10, step=1)
print(array)
grid = numpy.linspace(start=0, stop=10, num=11, endpoint=True)
print(grid)

### Build a logarithmic $k$-grid for power spectra

The matter power spectrum $P(k)$ is usually plotted over many orders of magnitude in wavenumber $k$. A logarithmic grid is therefore natural:

```python
k_grid = numpy.logspace(start=-4, stop=+2, num=100, endpoint=True)
```

This creates 100 points from $10^{-4}$ to $10^{2}\,\mathrm{Mpc}^{-1}$. We reuse this grid below and again in the power-spectrum assignment.

In [ ]:
k_grid = numpy.logspace(start=-4, stop=+2, num=100, endpoint=True)
print(k_grid)

---

## Turn the cosmology into matter power spectra

The **matter power spectrum** $P(k)$ describes how strongly matter density fluctuates on different spatial scales (wavenumber $k$).

- **Linear** $P(k)$: valid when density fluctuations are small. On large scales (small $k$), structure growth remains approximately linear.
- **Non-linear** $P(k)$: includes the effects of gravitational collapse on small scales (large $k$). At high $k$, the non-linear spectrum is enhanced relative to the linear one.

In PyCCL we use:

- `pyccl.power.linear_power(cosmo, k, a, ...)`
- `pyccl.power.nonlin_power(cosmo, k, a, ...)`

Here we evaluate both at redshift $z = 0$ (scale factor `a0 = 1`), using the same `k_grid` defined above. The argument `p_of_k_a='delta_matter:delta_matter'` asks for the matter–matter power spectrum.

The worked example below plots **linear as a solid line** and **non-linear as a dashed line**. In the power-spectrum assignment you will extend this idea to several redshifts and several colours.

### Redshift convention used by the IA notebooks

Later notebooks reuse each redshift grid through the pivot-normalized coordinate

$$
\boxed{r_\ast(z)=\frac{1+z}{1+z_\ast}}.
$$

This does not change the PyCCL scale factor $a=1/(1+z)$. It is only a convenient
dimensionless coordinate for the explicit IA factor
$R_z(z)=r_\ast^\eta(z)$ and for the modest evolution of $k_t(z)$ and $n(z)$.

In [ ]:
power_linear = pyccl.power.linear_power(
    cosmo=cosmology, 
    k=k_grid, 
    a=a0, 
    p_of_k_a='delta_matter:delta_matter'
)
print(power_linear)

In [ ]:
power_non_linear = pyccl.power.nonlin_power(
    cosmo=cosmology, 
    k=k_grid, 
    a=a0, 
    p_of_k_a='delta_matter:delta_matter'
)
print(power_non_linear)

In [ ]:

pyplot.rcParams['font.size'] = 25
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.family'] = 'Times New Roman'
figure, plot = pyplot.subplots(nrows=1, ncols=1, figsize=(10, 10))

plot.plot(
    k_grid, power_linear,
    linewidth=2, linestyle='-', color='black', rasterized=True,
    label=r'linear, $z=0$'
)

plot.plot(
    k_grid, power_non_linear,
    linewidth=2, linestyle='--', color='black', rasterized=True,
    label=r'non-linear, $z=0$'
)

plot.set_xscale('log')
plot.set_yscale('log')

plot.set_xlabel(r'$k$ [Mpc$^{-1}$]')
plot.set_ylabel(r'$P(k)$ [Mpc$^3$]')

plot.legend()
figure.savefig(figure_path / 'Matter_Power_Spectra.pdf', format='pdf', dpi=512, bbox_inches='tight')

---

# Put the workflow into practice

Complete the two tasks below in **new code cells** that you add after each task description. Do **not** overwrite the worked examples above — build on them.

---

## Explore cosmological distances across redshift

Plot the following distance measures as a function of redshift between $z = 0$ and $z = 10$:

1. **Comoving radial distance**
2. **Comoving transverse distance**
3. **Luminosity distance**
4. **Angular diameter distance**

### Suggested steps

1. Build a redshift array, for example with `numpy.linspace(0, 10, ...)`.
2. Convert redshifts to scale factors: `a = 1.0 / (1.0 + z)`.
3. Search the [PyCCL documentation](https://ccl.readthedocs.io/) for the distance functions that take `(cosmo, a)` as arguments. Useful keywords to search for: `comoving_radial_distance`, `comoving_transverse_distance`, `luminosity_distance`, `angular_diameter_distance`.
4. Evaluate each function on your scale-factor array (using the same `cosmology` object defined above).
5. Make **one figure** with all four curves vs $z$. Add axis labels and a legend.

### Hints

- Start from the $z = 0$ sanity-check cell and generalise it to an *array* of redshifts.
- In a flat Universe, the comoving radial and comoving transverse distances are equal — your plot should reflect that.
- At high redshift you should see $D_L > D_M > D_A$.

**Add your solution cell(s) immediately below this markdown cell.**

*← Student workspace for the distance assignment: add code cells here.*

In [ ]:
z = numpy.linspace(start=0, stop=10, num=100, endpoint=True)
a = (1.0 / (1.0 + z))

comoving_radial_distance = pyccl.background.comoving_radial_distance(cosmology, a)
comoving_angular_distance = pyccl.background.comoving_angular_distance(cosmology, a)
luminosity_distance = pyccl.background.luminosity_distance(cosmology, a)
angular_diameter_distance = pyccl.background.angular_diameter_distance(cosmology, a)

pyplot.rcParams['font.size'] = 25
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.family'] = 'Times New Roman'
figure, plot = pyplot.subplots(nrows=1, ncols=1, figsize=(10, 10))

plot.plot(
    z, comoving_radial_distance,
    linewidth=2, linestyle='-', color='purple', rasterized=True,
    label='comoving_radial_distance'
)

plot.plot(
    z, comoving_angular_distance,
    linewidth=2, linestyle='--', color='blue', rasterized=True,
    label='comoving_angular_distance'
)

plot.plot(
    z, luminosity_distance,
    linewidth=2, linestyle='--', color='orange', rasterized=True,
    label='luminosity_distance'
)

plot.plot(
    z, angular_diameter_distance,
    linewidth=2, linestyle='--', color='red', rasterized=True,
    label='angular_diameter_distance'
)

plot.set_xscale('log')
plot.set_yscale('log')

plot.set_xlabel(r'$z$')
plot.set_ylabel(r'$D$ [Mpc]')

plot.legend()
figure.savefig(figure_path / 'Cosmological_Distances.pdf', format='pdf', dpi=512, bbox_inches='tight')



---

## Compare matter power spectra at $z = 0$, $1$, and $2$

Extend the $z = 0$ power-spectrum example and compute **linear** and **non-linear** matter power spectra at **redshift $z = 1$ and $z = 2$** as well. Then plot **all six curves on a single figure**, together with the $z = 0$ results.

### Plotting requirements

| Style | Meaning |
|-------|---------|
| **Solid line** (`linestyle='-'`) | Linear $P(k)$ |
| **Dashed line** (`linestyle='--'`) | Non-linear $P(k)$ |
| **Different colours** | Different redshifts ($z = 0$, $1$, $2$) |

Also include:
- logarithmic axes for both $k$ and $P(k)$
- axis labels
- a clear legend

### Suggested steps

1. Reuse the existing `k_grid` (or recreate an identical one with `numpy.logspace`).
2. For each redshift $z \in \{0, 1, 2\}$, compute the scale factor `a = 1.0 / (1.0 + z)`.
3. Call `pyccl.power.linear_power(...)` and `pyccl.power.nonlin_power(...)` at each scale factor, exactly as in the worked example. Store the results in new arrays (e.g. `power_linear_z1`, `power_non_linear_z1`, …).
4. Create one `pyplot.subplots` figure and call `plot.plot(...)` six times (or loop over redshifts).
5. Choose three colours (one per redshift). Keep solid vs dashed to distinguish linear vs non-linear.

### Hints

- You do **not** need new CCL functions — the same `linear_power` / `nonlin_power` calls work at any redshift once `a` is set correctly.
- You may keep the $z = 0$ arrays already computed above and only add $z = 1$ and $z = 2$, then plot everything together.
- On large scales (small $k$), linear and non-linear curves should nearly overlap; the difference grows toward small scales (large $k$).
- At higher redshift, the overall amplitude of $P(k)$ is lower because structure has had less time to grow.

**Add your solution cell(s) immediately below this markdown cell.**

In [ ]:
k_grid = numpy.logspace(start=-4, stop=+2, num=100, endpoint=True)

z_list = [0, 1, 2]
color_list = ['blue', 'orange', 'red']

pyplot.rcParams['font.size'] = 25
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.family'] = 'Times New Roman'
figure, plot = pyplot.subplots(nrows=1, ncols=1, figsize=(10, 10))

for z, color in zip(z_list, color_list):
    a = (1.0 / (1.0 + z)) 
    power_linear = pyccl.power.linear_power(
        cosmo=cosmology, 
        k=k_grid, 
        a=a, 
        p_of_k_a='delta_matter:delta_matter'
    )
    power_non_linear = pyccl.power.nonlin_power(
        cosmo=cosmology, 
        k=k_grid, 
        a=a, 
        p_of_k_a='delta_matter:delta_matter'
    )
    
    plot.plot(
        k_grid, power_linear,
        linewidth=2, linestyle='-', color=color, rasterized=True,
        label=f'linear, $z={z}$'
    )
    
    plot.plot(
        k_grid, power_non_linear,
        linewidth=2, linestyle='--', color=color, rasterized=True,
        label=f'non-linear, $z={z}$'
    )

plot.set_xscale('log')
plot.set_yscale('log')

plot.set_xlabel(r'$k$ [Mpc$^{-1}$]')
plot.set_ylabel(r'$P(k)$ [Mpc$^3$]')

plot.legend()
figure.savefig(figure_path / 'Matter_Power_Spectra_Redshifts.pdf', format='pdf', dpi=512, bbox_inches='tight')


## Final consistency checks


In [ ]:
assert (data_path / "Planck.json").is_file()
assert figure_path.is_dir()
assert numpy.all(numpy.isfinite(power_linear))
assert numpy.all(numpy.isfinite(power_non_linear))
print("CCL notebook consistency checks passed.")
